# INSPIRE Mortality Pipeline — Run-Everything Notebook

This notebook is the runnable companion to **`research_questions_and_roadmap.md`**.
Each section below corresponds to a numbered section in that document, and runs the
actual code that answers it — the existing pipeline scripts (NELA, GBM, DNN transformer,
feature audits) plus a few new cells for the questions that didn't have runnable code yet
(the label-definition audit, the multi-operation sensitivity check, and a POSSUM
implementation to sit alongside NELA).

**Before running:** you need `inspire_subjects_small.zip` (the 30-patient development
subset — `survived/` + `died/` folders of per-patient JSON files). Everything here runs
against that subset. Where a section only becomes meaningful at full (99,886-patient)
scale, that's called out explicitly rather than pretended away.

**How to use this in Colab:** Runtime → Change runtime type → **T4 GPU** (only needed for
the DNN section, everything else is CPU-fast). Run cells top to bottom.

| # | Section here | Matches section in `research_questions_and_roadmap.md` |
|---|---|---|
| 1 | Setup | — |
| 2 | Load subjects | §1 |
| 3 | Label-definition audit | §2.1 |
| 4 | Multi-operation sensitivity | §2.2 |
| 5 | ASA | §2.3 |
| 6 | Feature & static-data audits | §2.13 items 1–2 |
| 7 | Feature selection | §2.5, §2.13 item 1 |
| 8 | HFRS frailty score | §2.11 |
| 9 | Clinical baselines: ASA / NELA / POSSUM / NEWS2 | §2.4, §2.13 item 6 |
| 10 | GBM baseline | Model 2, `docs/INSPIRE_Project_Notes.md` §3 |
| 11 | DNN transformer pipeline | Model 3, `docs/index.md` §8 |
| 12 | Results comparison table | `docs/index.md` §10 |
| 13 | Starter code for what's next | §3 (new research ideas) |


## 1. Setup

In [ ]:
# Clone the repo and move into src/
!git clone https://github.com/thrisharajkumar/inspire-analysis-thrisha.git
%cd inspire-analysis-thrisha/src


In [ ]:
# Install anything not already in Colab
!pip -q install sortedcontainers statsmodels scikit-learn --upgrade


In [ ]:
# Upload inspire_subjects_small.zip (the 30-patient dev subset), then extract it.
from google.colab import files
uploaded = files.upload()  # select inspire_subjects_small.zip


In [ ]:
import zipfile, os

zip_path = 'inspire_subjects_small.zip'
extract_dir = '/content/inspire_subjects_small'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

# handle the double-nested folder some exports have
contents = os.listdir(extract_dir)
if len(contents) == 1 and os.path.isdir(os.path.join(extract_dir, contents[0])):
    extract_dir = os.path.join(extract_dir, contents[0])

DATA_DIR = extract_dir
print('survived:', len(os.listdir(os.path.join(DATA_DIR, 'survived'))))
print('died:    ', len(os.listdir(os.path.join(DATA_DIR, 'died'))))
print('DATA_DIR =', DATA_DIR)


## 2. Load subjects
*(research doc §1)*

`subject.read_subjects()` walks every JSON file under `DATA_DIR` (it doesn't care that
they're split into `survived/`/`died/` subfolders) and returns real `Subject` objects with
all the methods used below — `.died()`, `.inhosp_death_30day()`, `.get_operations()`,
`.get_labs()`, etc.


In [ ]:
import subject as subject_module

subjects = subject_module.read_subjects(DATA_DIR)
print(f"Loaded {len(subjects)} subjects")

folder_label = {}
for label, folder in [(0, 'survived'), (1, 'died')]:
    for fn in os.listdir(os.path.join(DATA_DIR, folder)):
        if fn.endswith('.json'):
            folder_label[fn[:-5]] = label

print(f"folder labels found for {len(folder_label)} of {len(subjects)} loaded subjects")


## 3. Label-definition audit
*(research doc §1, §2.1 — the single highest-priority open issue in the project)*

Reproduces the check documented in `docs/index.md` §4: does the `survived/`/`died/`
**folder** the file sits in match `subject.died()` (died at any point) or
`subject.inhosp_death_30day()` (died within 30 days of the *last* operation)?


In [ ]:
import pandas as pd

rows = []
for sid, subj in subjects.items():
    rows.append({
        'subject_id': sid,
        'folder_label': folder_label.get(sid),
        'died_ever': subj.died(),
        'died_30day': subj.inhosp_death_30day(),
        'n_operations': len(subj.get_operations()),
    })
label_df = pd.DataFrame(rows)

match_died_ever = (label_df['folder_label'] == label_df['died_ever'].astype(int)).mean()
match_died_30day = (label_df['folder_label'] == label_df['died_30day'].astype(int)).mean()

print(f"folder_label vs died_ever  : {match_died_ever:.1%} match")
print(f"folder_label vs died_30day : {match_died_30day:.1%} match")

mismatch = label_df[(label_df['folder_label'] == 1) & (~label_df['died_30day'])]
print(f"\n{len(mismatch)} folder='died' patients did NOT die within 30 days of their last operation:")
mismatch[['subject_id', 'n_operations']]


In [ ]:
# The corrected label to use everywhere downstream in this notebook.
# (docs/index.md §4 decision: use died_30day(), not the raw folder name.)
true_label = {sid: int(subj.inhosp_death_30day()) for sid, subj in subjects.items()}

n_died = sum(true_label.values())
n_total = len(true_label)
print(f"Corrected labels: {n_died} died / {n_total - n_died} survived "
      f"({n_died/n_total:.1%} mortality) — pos_weight = {(n_total - n_died)/max(n_died,1):.2f}")


## 4. Multi-operation sensitivity check
*(research doc §2.2 — last-operation vs. first-operation vs. exclude)*

Computes the 30-day label three different ways for every multi-operation patient in the
subset, so you can see directly where the definitions disagree. On this 30-patient subset
this is a **mechanism check**, not a statistically meaningful comparison — re-run this
exact cell once the full 99,886-patient cohort is loaded to get a real answer.


In [ ]:
def died_30day_from_operation(subj, operation):
    inhosp_death_time = subj.get_inhosp_death_time()
    if inhosp_death_time is None:
        return False
    orout_time = int(operation['orout_time'].strip())
    return inhosp_death_time < orout_time + 30 * 24 * 60

multi_op_rows = []
for sid, subj in subjects.items():
    ops = subj.get_operations()
    if len(ops) < 2:
        continue
    first_op, last_op = ops[0], ops[-1]
    multi_op_rows.append({
        'subject_id': sid,
        'n_operations': len(ops),
        'label_from_last_op (current pipeline)': int(died_30day_from_operation(subj, last_op)),
        'label_from_first_op': int(died_30day_from_operation(subj, first_op)),
    })

multi_op_df = pd.DataFrame(multi_op_rows)
print(f"{len(multi_op_df)} of {len(subjects)} subjects in this subset have >1 operation")
if len(multi_op_df):
    disagree = multi_op_df[multi_op_df['label_from_last_op (current pipeline)']
                            != multi_op_df['label_from_first_op']]
    print(f"{len(disagree)} of those have a DIFFERENT label depending on first-op vs last-op")
multi_op_df


In [ ]:
# Three cohort definitions for the same underlying question -- run each through your
# model of choice later in this notebook and compare, per research doc §2.2's
# "treat as a sensitivity analysis, don't just pick one" recommendation.

single_op_ids = {sid for sid, subj in subjects.items() if len(subj.get_operations()) == 1}
print(f"Option 'exclude multi-op patients': {len(single_op_ids)} of {len(subjects)} remain")

# n.b. dropping multi-op patients is a selection-bias risk (research doc §2.2) --
# check whether they have a higher mortality rate than single-op patients before excluding:
multi_op_ids = set(subjects) - single_op_ids
if multi_op_ids:
    mr_single = pd.Series({sid: true_label[sid] for sid in single_op_ids}).mean()
    mr_multi = pd.Series({sid: true_label[sid] for sid in multi_op_ids}).mean()
    print(f"30-day mortality, single-op patients: {mr_single:.1%}")
    print(f"30-day mortality, multi-op patients:  {mr_multi:.1%}  <-- compare these before excluding anyone")


## 5. ASA physical status
*(research doc §2.3)*

`asa` lives on the operation record, not as a standalone field — pulled from the last
operation here. Not currently used anywhere in the DNN pipeline; this cell is the
starting point for adding it as a static input feature (research doc §2.5 step 5,
"static operative/demographic" modality).


In [ ]:
asa_rows = []
for sid, subj in subjects.items():
    last_op = subj.get_last_operation()
    asa_str = (last_op.get('asa') or '').strip()
    emop_str = (last_op.get('emop') or '').strip()
    asa_rows.append({
        'subject_id': sid,
        'asa': int(asa_str) if asa_str else None,
        'emop': int(emop_str) if emop_str else None,
        'died_30day': true_label[sid],
    })
asa_df = pd.DataFrame(asa_rows)

print("Mortality rate by ASA class (expect it to climb with class -- a sanity check on the data,")
print("matches docs/eda_findings.md §2 on the full cohort):")
asa_df.groupby('asa')['died_30day'].agg(['mean', 'count'])


## 6. Feature & static-data audits
*(research doc §2.13, items 1–2 — "already built, just needs running at scale")*

Runs `audit_features.py` exactly as documented in `docs/feature_audit_findings.md` §7,
against whatever's in `DATA_DIR`. Re-run this cell unchanged once you swap in the full
99,886-patient dataset — that upgrades every coverage number in
`feature_audit_findings.md` from "30-patient estimate" to real.

`docs/feature_audit_findings.md` also references a companion script,
`audit_static_categorical.py` (static operations facts + diagnoses + medications,
pre-op only) — it isn't in this export of the repo, so the cell after next reimplements
the same check inline, directly from the description in that doc, rather than skipping it.


In [ ]:
import os
os.environ['DATA_DIR'] = DATA_DIR


In [ ]:
!python audit_features.py "$DATA_DIR" parameters.csv


In [ ]:
# audit_static_categorical.py isn't in this export -- inline equivalent, following the
# description in docs/feature_audit_findings.md §7: static operation facts have 100%
# coverage by construction (one operation record per patient); diagnoses/medications are
# counted only if they occurred BEFORE orin_time, to avoid leaking post-op information
# (e.g. a diagnosis made BECAUSE OF the surgery) into a pre-op prediction.
static_rows = []
for sid, subj in subjects.items():
    last_op = subj.get_last_operation()
    orin_time = int(last_op['orin_time'].strip())

    n_diag_preop = sum(
        1 for d in subj.get_diagnoses()
        if d.get('chart_time', '').strip() and int(d['chart_time']) < orin_time
    )
    n_meds_preop = sum(
        1 for m in subj.get_medications()
        if m.get('chart_time', '').strip() and int(m['chart_time']) < orin_time
    )
    static_rows.append({
        'subject_id': sid,
        'age': last_op.get('age'),
        'sex': last_op.get('sex'),
        'asa': last_op.get('asa'),
        'emop': last_op.get('emop'),
        'department': last_op.get('department'),
        'n_diagnoses_preop': n_diag_preop,
        'n_medications_preop': n_meds_preop,
    })
static_df = pd.DataFrame(static_rows)
print(f"Static facts (age/sex/asa/emop/department): 100% coverage by construction, "
      f"{len(static_df)}/{len(static_df)} patients")
print(f"Median pre-op diagnosis count: {static_df['n_diagnoses_preop'].median():.0f}, "
      f"median pre-op medication count: {static_df['n_medications_preop'].median():.0f}")
static_df.head()


## 7. Feature selection
*(research doc §2.5, §2.13 item 1)*

`feature_selection_pipeline.py` already does exactly the "feature selection" step from
the research questions: univariate screen -> FDR correction for testing 54 features at
once -> drop redundant (highly correlated) features -> refit with department/age/ASA/emop
included, to check which features survive once you adjust for "this lab looks predictive
only because sicker departments order it more often."

The script's `SUBJECTS_DIR`/`PARAMS_CSV` constants default to a local Windows path — the
cells below import its functions directly and point them at `DATA_DIR` instead of editing
the file. **On this 30-patient subset, treat the output as a dry run of the mechanism,
not a trustworthy result** — `docs/feature_audit_findings.md` §8 already flags that
per-department counts this small (1-6 patients) aren't statistically meaningful yet.


In [ ]:
import feature_selection_pipeline as fsp

schema = fsp.load_schema('parameters.csv')
all_features = schema.get('labs', []) + schema.get('ward_vitals', [])

print('Step 1: loading patient data...')
fs_df = fsp.build_patient_table(DATA_DIR, all_features)
print(f"  {len(fs_df)} patients loaded ({fs_df.label.sum()} died, {(fs_df.label == 0).sum()} survived)")


In [ ]:
print('Step 2-3: univariate screen with FDR correction...')
res_df = fsp.univariate_screen(fs_df, all_features)
sig_features = res_df[res_df.p_value_fdr < 0.05].feature.tolist()
print(f"  {len(sig_features)} of {len(all_features)} features are FDR-significant")
res_df.sort_values('p_value').head(15)


In [ ]:
print('Step 4: removing redundant (highly correlated) features...')
pruned_features, dropped = fsp.drop_redundant_features(fs_df, sig_features, res_df)
print(f"  dropped as redundant: {sorted(dropped)}")
print(f"  {len(pruned_features)} features remain")

# also drop features with poor coverage
coverage = {f: max((fs_df[f"{f}__last"].notna() & (fs_df.label == 1)).mean(),
                    (fs_df[f"{f}__last"].notna() & (fs_df.label == 0)).mean())
            for f in pruned_features}
pruned_features = [f for f in pruned_features if coverage[f] >= 0.10]
print(f"  {len(pruned_features)} features remain after the coverage filter")


In [ ]:
zero_death_depts = fs_df.groupby('department').label.sum()
zero_death_depts = zero_death_depts[zero_death_depts == 0].index.tolist()
print(f"Step 5: fitting confound-adjusted model (excluding zero-death depts: {zero_death_depts})...")

try:
    result, summary, vif, reference_dept = fsp.fit_adjusted_model(fs_df, pruned_features, zero_death_depts)
    print(f"  reference department: {reference_dept}")
    print(f"  converged: {result.mle_retvals['converged']}, pseudo R2: {result.prsquared:.3f}")

    lab_summary = summary.loc[pruned_features].sort_values('p_value')
    final_features = lab_summary[lab_summary.p_value < 0.05].index.tolist()
    print(f"\n=== FINAL FEATURES (survive confound adjustment, p<0.05) ===")
    display(lab_summary.loc[final_features].round(4))
except Exception as e:
    print(f"Model fit failed on this small subset (expected with only "
          f"{int(fs_df.label.sum())} deaths total): {e}")
    print("Re-run this cell once the full-scale dataset is loaded.")


## 8. HFRS frailty score
*(research doc §2.11 — the existing worked example of "categorising ICD-10 codes into a
clinically meaningful score," used as the template for the new ICD-10 work)*

`compute_hfrs()` sums point-weights for 109 ICD-10 codes (Gilbert et al. 2018) present in
a patient's diagnosis history. **Known caveat, already flagged in the repo**
(`docs/eda_findings.md` §10): this implementation counts the patient's *entire* diagnosis
history, not the published 2-year window for patients 75+ — treat these numbers as
preliminary until that's fixed.


In [ ]:
import subject as subject_module

hfrs_rows = []
for sid, subj in subjects.items():
    score, category = subject_module.compute_hfrs(subj)
    hfrs_rows.append({'subject_id': sid, 'hfrs_score': score, 'hfrs_category': category,
                       'died_30day': true_label[sid]})
hfrs_df = pd.DataFrame(hfrs_rows)

print("Mortality rate by HFRS category:")
display(hfrs_df.groupby('hfrs_category')['died_30day'].agg(['mean', 'count']))
hfrs_df.sort_values('hfrs_score', ascending=False).head(10)


## 9. Clinical baselines: ASA, NELA, POSSUM, NEWS2
*(research doc §2.4 — "ASA, POSSUM, NELA, all differences clearly laid out")*

None of these four are machine-learned — they're fixed equations (or, for ASA, a
clinical judgement) developed once on external cohorts, included here specifically as the
non-learning baselines any model in this project has to beat. `nela.py` already
implements NELA. `score_models.py` already implements NEWS2 but it isn't compared to
anything elsewhere in the repo. **POSSUM was the one of the four not yet implemented
anywhere in this codebase — added below**, following the standard published equation
(Copeland et al. 1991; Portsmouth correction: Prytherch et al. 1998).


In [ ]:
import nela
import score_models

# --- NELA demo, using the repo's own worked example patient, 100033460 -----------------
# NELA needs several "centred" variables (deviation from the NELA derivation cohort's own
# mean, e.g. Age_cent = age - population_mean_age) that INSPIRE doesn't give us directly --
# research doc §2.4 flags this as a real external-validation question, not just a coding
# detail. This call demonstrates the mechanism with illustrative values, exactly as the
# `if __name__ == "__main__"` block at the bottom of nela.py does.
demo_risk = nela.compute_nela_score(
    Age_cent=5, ASA_3=1, Albumin=35,
    Pulse_cent=10, Pulse_cent2=100,
    SystolicBP_cent=-5, SystolicBP_cent2=25,
    LN_Urea_cent=0.3, LN_WBC_cent=0.2, LN_WBC_cent2=0.04,
    GCS_14=1, Malignancy_Primary=1, Respiratory_2=1,
    Urgency_2_6=1, Indication_Sepsis=1, Soiling_Severe=1,
)
print(f"NELA demo risk (illustrative inputs): {demo_risk:.4f}")


In [ ]:
def compute_possum_score(physiological_score, operative_severity_score, variant='possum'):
    """
    POSSUM / P-POSSUM mortality risk.

    physiological_score, operative_severity_score: sum of 12 (physiology) and 6
        (operative) POSSUM sub-scores, each individually scored 1/2/4/8 by severity.
        This notebook does not derive those sub-scores from INSPIRE fields (that mapping
        is itself a research task -- see docs/research_questions_and_roadmap.md §2.4) --
        pass them in directly, e.g. from a manual chart review or once a mapping exists.
    variant: 'possum' (original, Copeland et al. 1991) or 'p-possum'
        (Portsmouth correction, Prytherch et al. 1998 -- corrects POSSUM's tendency to
        overestimate mortality in low-risk patients).
    """
    import math
    ps, os_ = physiological_score, operative_severity_score
    if variant == 'possum':
        logit = -7.04 + 0.13 * ps + 0.16 * os_          # R1 = mortality risk (Copeland et al.)
    elif variant == 'p-possum':
        logit = -9.065 + 0.1692 * ps + 0.155 * os_      # Portsmouth correction
    else:
        raise ValueError("variant must be 'possum' or 'p-possum'")
    return 1 / (1 + math.exp(-logit))

# Demo, using the same illustrative severity level as the NELA example above:
print(f"POSSUM   demo risk (PS=20, OS=16): {compute_possum_score(20, 16, 'possum'):.4f}")
print(f"P-POSSUM demo risk (PS=20, OS=16): {compute_possum_score(20, 16, 'p-possum'):.4f}")


In [ ]:
# NEWS2 demo -- score_models.py already has this, unused elsewhere in the repo.
# It's a vitals-only, deterioration-focused score (built for ward monitoring), a genuinely
# different kind of tool from the pre-op-risk-focused ASA/POSSUM/NELA above -- worth
# deciding whether it belongs in the same comparison table or a separate one.
news2_demo = score_models.compute_news2_score(
    resp_rate=22, spo2=94, temp=38.2, systolic_bp=105,
    heart_rate=115, consciousness='A', oxygen_support=False,
)
print(f"NEWS2 demo score: {news2_demo}")


## 10. GBM baseline (Model 2)
*(`docs/INSPIRE_Project_Notes.md` §3 — "Saranya's model")*

Builds the same 18-feature pre-op snapshot table `gbm_mortality_pipeline.py` builds
(demographics + most-recent pre-op labs), reusing the `subjects` already loaded above
instead of re-reading from disk. Fits a plain Logistic Regression, the same baseline
comparator the original script uses (full XGBoost needs more data than this 30-patient
subset can support meaningfully — swap `LogisticRegression` for
`xgboost.XGBClassifier` once running at full scale, exactly as `docs/INSPIRE_Project_Notes.md`
§3 describes).


In [ ]:
from gbm_mortality_pipeline import calculate_bmi

INPUT_VARS = ['age', 'sex', 'emop', 'bmi', 'andur',
              'preop_hb', 'preop_platelet', 'preop_wbc',
              'preop_aptt', 'preop_ptinr', 'preop_glucose',
              'preop_bun', 'preop_albumin', 'preop_ast',
              'preop_alt', 'preop_creatinine', 'preop_sodium',
              'preop_potassium']

gbm_rows = {}
for sid, subj in subjects.items():
    last_op = subj.get_last_operation()
    row = dict(last_op)
    row['age'] = int(row['age'])
    row['sex'] = (row['sex'] == 'M')
    row['emop'] = int(row['emop']) if row.get('emop', '').strip() else None
    asa_str = row.get('asa', '').strip()
    row['asa'] = int(asa_str) if asa_str else None

    orin_time = int(row['orin_time'].strip())
    for lab in ['hb', 'platelet', 'aptt', 'wbc', 'ptinr', 'glucose', 'bun',
                'albumin', 'ast', 'alt', 'creatinine', 'sodium', 'potassium']:
        row[f'preop_{lab}'] = subj.get_most_recent_lab(lab, orin_time)

    anend, anstart = row.get('anend_time', '').strip(), row.get('anstart_time', '').strip()
    row['andur'] = (int(anend) - int(anstart)) if anend and anstart else None

    weight_str, height_str = row.get('weight', '').strip(), row.get('height', '').strip()
    row['bmi'] = calculate_bmi(float(weight_str), float(height_str) / 100.0) \
                 if weight_str and height_str and float(weight_str) > 0 and float(height_str) > 0 else None

    row['inhosp_death_30day'] = int(subj.inhosp_death_30day())
    gbm_rows[sid] = row

gbm_df = pd.DataFrame.from_dict(gbm_rows, orient='index')
print(gbm_df[INPUT_VARS + ['inhosp_death_30day']].shape)
gbm_df[INPUT_VARS + ['inhosp_death_30day']].head()


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

X = gbm_df[INPUT_VARS].astype(float)
y = gbm_df['inhosp_death_30day']

X_imputed = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X), columns=X.columns)

try:
    X_train, X_test, y_train, y_test = train_test_split(
        X_imputed, y, test_size=0.33, random_state=42, stratify=y)
    gbm_baseline = LogisticRegression(max_iter=1000, class_weight='balanced')
    gbm_baseline.fit(X_train, y_train)
    gbm_auroc = roc_auc_score(y_test, gbm_baseline.predict_proba(X_test)[:, 1])
    print(f"GBM baseline (Logistic Regression, 18 pre-op features) AUROC: {gbm_auroc:.4f}")
except ValueError as e:
    print(f"Not enough deaths in this small subset to split/stratify cleanly: {e}")
    print("Fit on the full data instead, evaluate on training data only as a sanity check:")
    gbm_baseline = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_imputed, y)
    gbm_auroc = roc_auc_score(y, gbm_baseline.predict_proba(X_imputed)[:, 1])
    print(f"GBM baseline (in-sample) AUROC: {gbm_auroc:.4f}  <-- optimistic, re-run properly at full scale")


## 11. DNN transformer pipeline (Model 3 — the project's main contribution)
*(`docs/index.md` §8 — the two-phase pipeline: autoencoder pre-training, then classifier
fine-tuning)*

Runs `dnn_mortality_pipeline.py` end to end, exactly as documented in `docs/index.md`
§12's Colab workflow, using the `DATA_DIR` already extracted above. This needs a GPU
runtime to be reasonably fast (Runtime → Change runtime type → T4 GPU) but will also run,
more slowly, on CPU.


In [ ]:
import importlib
import dnn_mortality_data
import dnn_mortality_pipeline as pipeline
importlib.reload(dnn_mortality_data)
importlib.reload(pipeline)

FEATURE_COLUMNS = ['glucose', 'potassium', 'sodium', 'creatinine', 'hr', 'spo2', 'nibp_sbp']

subjects_data, seq_length = dnn_mortality_data.load_real_subjects(
    DATA_DIR, FEATURE_COLUMNS, days_before_operation=5
)

num_died = sum(1 for s in subjects_data.values() if s['label'] == 1)
num_survived = sum(1 for s in subjects_data.values() if s['label'] == 0)
total = num_died + num_survived
not_survived_pct = num_died / total if total else 0.5
print(f"num_died={num_died}  num_survived={num_survived}  mortality={not_survived_pct:.1%}")


In [ ]:
max_length = 0
for subject in subjects_data.values():
    df = pipeline.align_time_series(subject['timeseries'])
    max_length = max(max_length, len(df))
global_length = min(max_length, 1440)
print(f"global_length = {global_length}")

train_data, test_data = pipeline.create_train_test_split(
    subjects_data, train_size=2 / 3, not_survived_pct=not_survived_pct
)
print(f"train subjects = {len(train_data)}, test subjects = {len(test_data)}")


In [ ]:
# Phase 1: autoencoder pre-training (unsupervised, no mortality labels used)
auto_dataloader, scaler, num_features = pipeline.preprocess_for_autoencode(
    train_data, seq_length, FEATURE_COLUMNS)

device = pipeline.get_device()
print(f"device = {device}, num_features = {num_features} (feature + mask columns)")

autoencoder = pipeline.train_autoencoder(auto_dataloader, num_features, device, epochs=10)


In [ ]:
# Embeddings + t-SNE, before classifier fine-tuning (docs/INSPIRE_Project_Notes.md §10)
mask_columns = [f'{col}_mask' for col in FEATURE_COLUMNS]
train_dataset = pipeline.SubjectDataset(train_data, scaler, global_length, FEATURE_COLUMNS, mask_columns, num_features)
test_dataset = pipeline.SubjectDataset(test_data, scaler, global_length, FEATURE_COLUMNS, mask_columns, num_features)

from torch.utils.data import DataLoader
train_batch_size = min(32, max(1, len(train_dataset)))
test_batch_size = min(32, max(1, len(test_dataset)))
train_dataloader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False)

embeddings, emb_labels = pipeline.extract_embeddings(autoencoder, test_dataloader, device)
pipeline.visualise_embeddings(embeddings, emb_labels)


In [ ]:
# Phase 2: classifier fine-tuning (encoder unfrozen -- docs/index.md §13, the
# "frozen encoder gives ~0.47 for everyone" bug fix already applied here)
train_num_survived = sum(1 for p in train_data.values() if p['label'] == 0)
train_num_died = sum(1 for p in train_data.values() if p['label'] == 1)
pos_weight = (train_num_survived / train_num_died) if train_num_died > 0 else 1.0
print(f"pos_weight = {pos_weight:.2f}")

classifier = pipeline.train_classifier(
    autoencoder, train_dataloader, device, epochs=20, pos_weight=pos_weight, unfreeze_encoder=True
)


In [ ]:
# evaluate_model() prints AUROC, best F1/threshold, and test-set composition, and saves
# auroc.png / auprc.png (docs/index.md §8) -- it returns the AUROC value only.
dnn_auroc = pipeline.evaluate_model(classifier, test_dataloader, device)
print(f"\nDNN transformer test AUROC: {dnn_auroc:.4f}")


## 12. Results comparison table
*(`docs/index.md` §10 — collects everything run above into one place)*

Remember, on this 30-patient development subset, every number below is noisy by
construction (`docs/index.md` §10: "one wrong prediction moves AUROC by ~0.11") — the
point of this table is to have the comparison wired up and ready, not to draw conclusions
from it yet. Re-run the whole notebook against the full 99,886-patient cohort for numbers
worth reporting.


In [ ]:
results = pd.DataFrame([
    {'model': 'NELA (demo values, not fit to INSPIRE)', 'AUROC': None, 'notes': 'fixed clinical equation, no training -- see §9'},
    {'model': 'POSSUM (demo values, not fit to INSPIRE)', 'AUROC': None, 'notes': 'fixed clinical equation, newly added -- see §9'},
    {'model': 'GBM baseline (Logistic Regression, 18 pre-op features)', 'AUROC': round(gbm_auroc, 4), 'notes': 'see §10'},
    {'model': 'DNN transformer (two-phase, 7 features)', 'AUROC': round(dnn_auroc, 4), 'notes': 'see §11'},
    {'model': 'Shickel et al. 2023 (published benchmark, 56,242 patients)', 'AUROC': 0.92, 'notes': 'the number to beat, docs/index.md §16'},
])
results


## 13. Starter code for what's next
*(research doc §3 — new research ideas; not full implementations, but working scaffolding
to build on, using data already loaded in this notebook)*


### 13a. Organ-system feature grouping
*(research doc §2.5, §2.12 — the target architecture in your diagram)*

The grouping from `docs/index.md` §15 item 4, as a dict — this is the thing to loop over
once you split the single `TimeSeriesTransformer` into one encoder per system.


In [ ]:
ORGAN_SYSTEMS = {
    'renal': {
        'labs': ['bun', 'calcium', 'chloride', 'creatinine', 'ica', 'phosphorus', 'potassium', 'sodium'],
        'ward_vitals': ['crrt', 'uo'],
    },
    'cardiovascular': {
        'labs': ['ck', 'ckmb', 'troponin_i', 'troponin_t'],
        'ward_vitals': ['hr', 'nibp_sbp', 'nibp_dbp', 'nibp_mbp', 'iabp'],
    },
    'respiratory': {
        'labs': ['be', 'hco3', 'paco2', 'pao2', 'ph', 'sao2'],
        'ward_vitals': ['fio2', 'rr', 'spo2', 'vent', 'ecmo'],
    },
    'metabolic_hepatic': {
        'labs': ['albumin', 'alp', 'alt', 'ast', 'glucose', 'hba1c', 'lacate', 'total_bilirubin', 'total_protein'],
        'ward_vitals': ['bt'],
    },
    'haematology_coagulation': {
        'labs': ['aptt', 'crp', 'd_dimer', 'fibrinogen', 'hb', 'hct', 'lymphocyte', 'platelet', 'ptinr', 'seg', 'wbc'],
        'ward_vitals': [],
    },
    'neurological': {
        'labs': [],
        'ward_vitals': ['gcs_e', 'gcs_m', 'gcs_v'],
    },
}

# Sketch: one TimeSeriesTransformer instance per system, reusing the existing pipeline
# functions unchanged -- just called once per system with that system's feature list.
# system_encoders = {}
# for system_name, feats in ORGAN_SYSTEMS.items():
#     feature_columns = feats['labs'] + feats['ward_vitals']
#     if not feature_columns:
#         continue
#     subjects_data_sys, seq_length_sys = dnn_mortality_data.load_real_subjects(
#         DATA_DIR, feature_columns, days_before_operation=5)
#     auto_dl, scaler_sys, num_feat_sys = pipeline.preprocess_for_autoencode(
#         subjects_data_sys, seq_length_sys, feature_columns)
#     system_encoders[system_name] = pipeline.train_autoencoder(auto_dl, num_feat_sys, device, epochs=10)
# -- each system_encoders[name] then produces that system's embedding (research doc §2.5 step 3);
#    fusing them (research doc §2.9) is the next piece of new code to write, not yet here.


### 13b. Attention audit scaffold
*(research doc §2.14C item 4 — genuine mechanistic evidence, not a post-hoc explanation)*

`TimeSeriesTransformer`'s encoder layers don't expose attention weights by default with
`batch_first=True` PyTorch `TransformerEncoder`. Getting them out requires either a
forward hook or rebuilding the encoder loop manually with
`need_weights=True` on each `MultiheadAttention` call — left as a TODO here rather than
silently faked, since it needs a small change to `TimeSeriesTransformer.forward()` in
`dnn_mortality_pipeline.py` (add an optional `return_attention=True` path) before it can
run. Once available, the check described in the research doc is: do attention weights
spike at the chart-times of the acute-deterioration diagnoses already found in
`docs/eda_findings.md` §4 (D65, I46, R57, J80, K72, A41)?


In [ ]:
# TODO (research doc §2.14C item 4): once TimeSeriesTransformer.forward() can return
# per-head attention weights, this is where to plot them against a patient's diagnosis
# timeline. Left unimplemented deliberately -- see markdown cell above for why.


### 13c. Calibration check
*(research doc §2.14D — "AUROC tells you ranking, not whether 0.3 means 30%")*

A calibration curve for whichever probability outputs you already have above (GBM
baseline used here as the example — swap in `classifier`'s predicted probabilities for
the DNN once you have them in a plain array).


In [ ]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

probs = gbm_baseline.predict_proba(X_imputed)[:, 1]
prob_true, prob_pred = calibration_curve(y, probs, n_bins=5, strategy='quantile')

plt.figure(figsize=(5, 5))
plt.plot(prob_pred, prob_true, marker='o', label='GBM baseline')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='perfectly calibrated')
plt.xlabel('Predicted probability')
plt.ylabel('Observed frequency')
plt.title('Calibration curve (5 bins -- widen once N is larger)')
plt.legend()
plt.show()


---

## Where to go from here

Everything above is either an existing script wired up to run end to end, or a small new
addition (the label audit, the multi-op sensitivity check, the POSSUM implementation).
The bigger pieces of new work — the system-separated architecture, the time-to-event
reframing, the Concept Bottleneck / sparse-autoencoder / prototype interpretability layers
— are design work first, code second; they're laid out in full in
`docs/research_questions_and_roadmap.md` §2.5–§2.14 and prioritised in its §4 roadmap. The
single next action, before any of that: fix the label-definition bug in Section 3 of this
notebook and re-run everything above against the full 99,886-patient cohort once it's
available.
